# 11: Autograd Magic - Automatic Differentiation

## No More Manual Backprop!

Remember writing all those gradient calculations by hand? PyTorch's **autograd** does it automatically!

### The Web Dev Analogy

Autograd is like **hot module reloading**:
- You make changes (forward pass)
- The system automatically figures out what to update (backward pass)
- No manual configuration needed

## What You'll Learn
- [ ] Use PyTorch's autograd to compute gradients automatically
- [ ] Explain computational graphs and how they track operations
- [ ] Compare manual gradient computation vs automatic differentiation

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 9**: Manual backpropagation (chain rule by hand) | Autograd does all that work *automatically* — no more manual derivatives! |
| **Lesson 10**: PyTorch tensors | Tensors with `requires_grad=True` build a computational graph as you compute |

In [ ]:
import torch
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

print("Ready to explore autograd! ✨")

## 1. The requires_grad Flag

Tensors with `requires_grad=True` track all operations for automatic differentiation:

In [ ]:
# Regular tensor - no gradient tracking
x = torch.tensor([1., 2., 3.])
print(f"Regular tensor: requires_grad = {x.requires_grad}")

# Tensor that tracks gradients
x = torch.tensor([1., 2., 3.], requires_grad=True)
print(f"Gradient tensor: requires_grad = {x.requires_grad}")

# Or enable it later
y = torch.tensor([1., 2., 3.])
y.requires_grad_(True)  # In-place operation (note the underscore)
print(f"After enabling: requires_grad = {y.requires_grad}")

## 2. Simple Gradient Computation

In [ ]:
# Simple example: y = x^2
x = torch.tensor([2.0], requires_grad=True)

# Forward pass
y = x ** 2
print(f"x = {x.item()}")
print(f"y = x² = {y.item()}")

# Backward pass - compute gradients
y.backward()

# dy/dx = 2x = 2*2 = 4
print(f"\ndy/dx = {x.grad.item()}")
print(f"Expected: 2x = 2*2 = 4 ✓")

In [ ]:
# More complex: y = x^3 + 2x^2 - 5x
x = torch.tensor([3.0], requires_grad=True)

y = x**3 + 2*x**2 - 5*x
print(f"y = x³ + 2x² - 5x")
print(f"At x = {x.item()}: y = {y.item()}")

y.backward()

# dy/dx = 3x² + 4x - 5 = 3*9 + 4*3 - 5 = 27 + 12 - 5 = 34
print(f"\ndy/dx = 3x² + 4x - 5 = {x.grad.item()}")
expected = 3*3**2 + 4*3 - 5
print(f"Expected: {expected} ✓")

## 3. Gradients with Multiple Variables

In [ ]:
# z = f(x, y) = x*y + x^2
x = torch.tensor([2.0], requires_grad=True)
y = torch.tensor([3.0], requires_grad=True)

z = x * y + x**2
print(f"z = x*y + x²")
print(f"x = {x.item()}, y = {y.item()}")
print(f"z = {z.item()}")

z.backward()

# ∂z/∂x = y + 2x = 3 + 4 = 7
# ∂z/∂y = x = 2
print(f"\n∂z/∂x = y + 2x = {x.grad.item()}")
print(f"∂z/∂y = x = {y.grad.item()}")

## 4. Computational Graphs

PyTorch builds a graph of all operations. Let's visualize:

In [ ]:
# The computation graph tracks operations
a = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([3.0], requires_grad=True)

c = a + b       # AddBackward
d = a * b       # MulBackward
e = c * d       # MulBackward

print(f"a = {a.item()}, b = {b.item()}")
print(f"c = a + b = {c.item()}")
print(f"d = a * b = {d.item()}")
print(f"e = c * d = {e.item()}")

# Check the grad_fn (what created this tensor)
print(f"\nc.grad_fn: {c.grad_fn}")
print(f"d.grad_fn: {d.grad_fn}")
print(f"e.grad_fn: {e.grad_fn}")

# Compute gradients
e.backward()
print(f"\n∂e/∂a = {a.grad.item()}")
print(f"∂e/∂b = {b.grad.item()}")

## 5. Gradient for Neural Network Weights

In [ ]:
# Simulate a simple neural network
torch.manual_seed(42)

# Input and target
X = torch.tensor([[1.0, 2.0]])  # 1 sample, 2 features
y = torch.tensor([[1.0]])        # Target

# Weights (these are what we want to learn!)
W = torch.randn(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

print(f"Initial weights W:\n{W}")
print(f"Initial bias b: {b}")

# Forward pass
y_pred = X @ W + b
loss = (y_pred - y) ** 2  # MSE loss

print(f"\nPrediction: {y_pred.item():.4f}")
print(f"Target: {y.item()}")
print(f"Loss: {loss.item():.4f}")

# Backward pass
loss.backward()

print(f"\nGradients:")
print(f"∂L/∂W = {W.grad.flatten().tolist()}")
print(f"∂L/∂b = {b.grad.item():.4f}")

In [ ]:
# Manual gradient descent step
learning_rate = 0.1

# Update weights (notice no_grad context!)
with torch.no_grad():
    W -= learning_rate * W.grad
    b -= learning_rate * b.grad

print(f"Updated weights W:\n{W}")
print(f"Updated bias b: {b}")

# New prediction
y_pred_new = X @ W + b
loss_new = (y_pred_new - y) ** 2

print(f"\nNew prediction: {y_pred_new.item():.4f}")
print(f"New loss: {loss_new.item():.4f}")
print(f"\n✅ Loss decreased!")

## 6. Important: Zeroing Gradients

In [ ]:
# Gradients ACCUMULATE by default!
x = torch.tensor([2.0], requires_grad=True)

# First backward
y1 = x ** 2
y1.backward()
print(f"After 1st backward: x.grad = {x.grad.item()}")

# Second backward (WITHOUT zeroing)
y2 = x ** 2
y2.backward()
print(f"After 2nd backward: x.grad = {x.grad.item()}  (accumulated!)")

# Third backward WITH zeroing
x.grad.zero_()  # Zero the gradient!
y3 = x ** 2
y3.backward()
print(f"After zeroing + 3rd backward: x.grad = {x.grad.item()}  (correct!)")

print("\n⚠️  Always zero gradients before each training iteration!")

## 7. Detaching from the Graph

In [ ]:
# Sometimes you want to stop gradient tracking
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2

# y still tracks gradients
print(f"y.requires_grad: {y.requires_grad}")

# Detach - creates a new tensor without gradient
y_detached = y.detach()
print(f"y_detached.requires_grad: {y_detached.requires_grad}")

# Use with torch.no_grad() for inference
with torch.no_grad():
    z = x ** 2
    print(f"Inside no_grad: z.requires_grad = {z.requires_grad}")

## 8. Complete Training Example

In [ ]:
# Linear regression with autograd
torch.manual_seed(42)

# Generate data: y = 3x + 2 + noise
X = torch.linspace(0, 10, 100).reshape(-1, 1)
y = 3 * X + 2 + torch.randn_like(X) * 0.5

# Initialize parameters
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.01
losses = []

print("Training linear regression...")
for epoch in range(100):
    # Forward pass
    y_pred = X * w + b
    loss = ((y_pred - y) ** 2).mean()  # MSE
    losses.append(loss.item())
    
    # Backward pass
    loss.backward()
    
    # Update (no gradient tracking here!)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
    
    # Zero gradients!
    w.grad.zero_()
    b.grad.zero_()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}, w = {w.item():.3f}, b = {b.item():.3f}")

print(f"\n✅ Final: w = {w.item():.3f} (true: 3), b = {b.item():.3f} (true: 2)")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

# Fit
with torch.no_grad():
    y_pred = X * w + b

axes[1].scatter(X.numpy(), y.numpy(), alpha=0.5, label='Data')
axes[1].plot(X.numpy(), y_pred.numpy(), 'r-', linewidth=2, label='Learned fit')
axes[1].set_xlabel('X')
axes[1].set_ylabel('y')
axes[1].set_title('Linear Regression Fit')
axes[1].legend()

plt.tight_layout()
plt.show()

## 📝 Check Your Understanding

1. What does `requires_grad=True` do?
2. Why do we need `torch.no_grad()` when updating weights?
3. Why must we zero gradients between iterations?
4. What does `.backward()` compute?
5. What's the purpose of `.detach()`?

In [ ]:
# --- Exercise 1: Compute Gradient with Autograd ---
# Create x = 3.0 with requires_grad=True, compute y = x² + 2x + 1,
# call y.backward(), and check x.grad.
# Analytical: dy/dx = 2x + 2 = 8.0

# YOUR CODE HERE:
x = torch.tensor(3.0, requires_grad=True)
y = None  # Compute x**2 + 2*x + 1
# Call y.backward() to compute gradients
grad_value = None  # Read x.grad.item()

# --- Check ---
assert y is not None, "Compute y = x² + 2x + 1!"
assert grad_value is not None, "Read the gradient from x.grad!"
assert abs(grad_value - 8.0) < 0.001, f"dy/dx = 2×3 + 2 = 8.0, got {grad_value}"
print(f"Exercise 1 passed! ✓  (dy/dx at x=3: {grad_value})")

# --- Quick Check: Why zero_grad()? ---
# Why must we call optimizer.zero_grad() (or zero gradients) before each backward pass?
# a) To free up GPU memory
# b) Because PyTorch accumulates gradients — they'd add up across iterations
# c) To reset the learning rate
# d) It's optional, just a best practice

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "PyTorch ADDS new gradients to existing ones. Without zeroing, gradients from previous steps accumulate!"
print("Exercise 2 passed! ✓")

# --- Exercise 3: Gradient of sin(x) ---
# Compute d/dx sin(x) at x = 0 using autograd.
# Analytical: cos(0) = 1.0

# YOUR CODE HERE:
x2 = torch.tensor(0.0, requires_grad=True)
y2 = None  # Compute torch.sin(x2)
# Call backward and read gradient
sin_grad = None  # x2.grad.item()

# --- Check ---
assert sin_grad is not None, "Compute the gradient!"
assert abs(sin_grad - 1.0) < 0.001, f"d/dx sin(x) at x=0 = cos(0) = 1.0, got {sin_grad}"
print("Exercise 3 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

Autograd handles all gradient computation:
1. **Forward pass**: Build computation graph
2. **`.backward()`**: Compute all gradients automatically
3. **`.grad`**: Access the computed gradients
4. **`.zero_()`**: Clear gradients for next iteration

Key patterns:
- `requires_grad=True` for learnable parameters
- `with torch.no_grad():` for inference/updates
- Always zero gradients between training steps!

**Next up**: The complete PyTorch training loop! →